## Pranay's Contribution

## llm prompts
- How to get the huggingface token?
- which model would be good at this task as per my project i mentioned
- how to avoid the version errors
- can you use the maximum gpu in the colab to run the process fast?

Model initialization

In [4]:
!pip install -q bitsandbytes faiss-cpu sentence-transformers langchain datasets transformers accelerate huggingface_hub
!pip install -q langchain-community


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 905.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 111.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 598.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import pandas as pd

# Load BioASQ biomedical passages
passages_df = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")

# Load test set (questions + answers)
test_df = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
from sentence_transformers import SentenceTransformer
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.schema import Document

# Convert passage rows into LangChain Documents
documents = [Document(page_content=row["passage"]) for _, row in passages_df.iterrows()]

# Embed with MiniLM
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Build FAISS vector store
vectorstore = FAISS.from_documents(documents, embedding_model)


/tmp/ipython-input-4-1094568553.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain.llms import HuggingFacePipeline
import torch

# Step 4: Load TinyLlama model
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.1
)

llm = HuggingFacePipeline(pipeline=pipe)

Device set to use cuda:0
/tmp/ipython-input-7-2869823758.py:24: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [10]:
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA


prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a biomedical question-answering assistant.
Only answer biomedical-related questions using the context provided.
If the question is unrelated to biomedicine (e.g., about economics, sports, politics), say:
"I'm only able to answer biomedical-related questions."

Context:
{context}

Question:
{question}

Answer:
"""
)

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(search_type="similarity", k=10),
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt_template}
)


In [11]:
# Step 6: Define the RAG chain
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(search_type="similarity", k=10),
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt_template}
)


In [12]:
# Step 7: Add context validation
from sentence_transformers import SentenceTransformer, util

# Load once
similarity_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def is_context_relevant(question, docs, threshold=0.3):
    q_embed = similarity_model.encode(question, convert_to_tensor=True)
    for doc in docs:
        d_embed = similarity_model.encode(doc.page_content, convert_to_tensor=True)
        score = util.pytorch_cos_sim(q_embed, d_embed).item()
        if score > threshold:
            return True
    return False

In [13]:
# Limit context length
def truncate_docs(docs, max_chars=1800):
    total = ""
    for doc in docs:
        if len(total) + len(doc.page_content) > max_chars:
            break
        total += doc.page_content + "\n"
    return total


In [14]:
def ask_question(query):
    docs = vectorstore.similarity_search(query, k=10)
    if is_context_relevant(query, docs):
        context = truncate_docs(docs)
        prompt = prompt_template.format(context=context, question=query)
        print("Answer:", llm(prompt))
    else:
        print("Answer: I'm only able to answer biomedical-related questions.")


In [15]:
# Step 9: Try it out

ask_question("What is the effect of diabetes")

/tmp/ipython-input-14-1538614172.py:6: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  print("Answer:", llm(prompt))


Answer: 
You are a biomedical question-answering assistant.
Only answer biomedical-related questions using the context provided.
If the question is unrelated to biomedicine (e.g., about economics, sports, politics), say:
"I'm only able to answer biomedical-related questions."

Context:
Metabolic syndrome is a cluster of conditions that synergistically increase the 
risk of cardiovascular disease, type 2 diabetes, and premature mortality. The 
components are abdominal obesity, impaired glucose metabolism, dyslipidemia, and 
hypertension. Prediabetes, which is a combination of excess body fat and insulin 
resistance, is considered an underlying etiology of metabolic syndrome. 
Prediabetes manifests as impaired fasting glucose and/or impaired glucose 
tolerance. Impaired fasting glucose is defined as a fasting blood glucose level 
of 100 to 125 mg/dL; impaired glucose tolerance requires a blood glucose level 
of 140 to 199 mg/dL 2 hours after a 75-g oral intake of glucose. In patients 
wi

In [20]:
ask_question("What is the politics")

Answer: I'm only able to answer biomedical-related questions.
